# NonlineRegressor — scikit-learn interface

`dt.NonlineRegressor` wraps the LSI / EDP / DSB methods behind the standard
`fit` / `predict` / `score` API and composes with sklearn `Pipeline`,
`GridSearchCV` and `cross_val_score`.

In [1]:
import numpy as np
import dtfit as dt
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold

## Data

In [2]:
rng = np.random.default_rng(0)
x = np.linspace(0, 10, 400)
y = 5.0 * np.arctan(1.5 * x) + rng.normal(0, 0.3, x.size)

## fit / predict / score

In [3]:
reg = dt.NonlineRegressor("a*atan(w*x)", "x", method="edp", p0=[1.0, 1.0]).fit(x, y)
print("coef_:", reg.coef_, "  R^2:", reg.score(x, y))
print("predict[:3]:", reg.predict(x[:3]))

coef_: [4.97159133 1.55206774]   R^2: 0.9577938837064025
predict[:3]: [0.         0.19329218 0.38600177]


## sklearn Pipeline

In [4]:
pipe = Pipeline([("reg", dt.NonlineRegressor("a*atan(w*x)", "x", method="edp", p0=[1.0, 1.0]))])
pipe.fit(x.reshape(-1, 1), y)
print("pipeline R^2:", pipe.score(x.reshape(-1, 1), y))

pipeline R^2: 0.9577938837064025


## GridSearchCV + cross_val_score
Use shuffled folds — contiguous folds split a monotonic curve into disjoint domains and depress scores for area-based methods.

In [5]:
gs = GridSearchCV(
    dt.NonlineRegressor("a*atan(w*x)", "x", method="edp", p0=[1.0, 1.0]),
    {"active_ratio": [0.5, 0.7, 0.9]},
    cv=KFold(3, shuffle=True, random_state=0),
).fit(x.reshape(-1, 1), y)
print("best params:", gs.best_params_, " best R^2:", gs.best_score_)

cv = cross_val_score(
    dt.NonlineRegressor("a*atan(w*x)", "x", method="edp", p0=[1.0, 1.0]),
    x.reshape(-1, 1), y, cv=KFold(3, shuffle=True, random_state=0),
)
print("cross_val_score:", cv)

best params: {'active_ratio': 0.7}  best R^2: 0.9540544046741749
cross_val_score: [0.96566713 0.95529536 0.94073392]
